# 📉 Module 7: PCA — Dimensionality Reduction
## Theory + Practical

---

## 📚 THEORY

**Problem:** When you have many features, data becomes hard to visualize, slow to train, and can overfit.

**PCA (Principal Component Analysis)** finds new axes (principal components) that capture the **most variance** in the data.

```
Original features (100 columns)
        ↓  PCA
2-3 principal components  (still captures 95% of information!)
```

### How PCA Works
1. Center the data (subtract mean)
2. Compute **Covariance Matrix**
3. Find **Eigenvectors** (directions of maximum variance) and **Eigenvalues** (how much variance each captures)
4. Sort by eigenvalue — top components = most important directions
5. Project data onto top K components

### Key Term: Explained Variance Ratio
- How much of the total variance does each component explain?
- Choose components that together explain ≥ 95% of variance

---

## 🔬 PRACTICAL

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_iris, load_digits

np.random.seed(42)
print('Libraries loaded ✅')

### Practical 1: PCA Intuition — 2D to 1D

In [ ]:
# Create correlated 2D data
mean = [0, 0]
cov = [[3, 2.5], [2.5, 3]]
X_2d = np.random.multivariate_normal(mean, cov, 200)

# Apply PCA
pca = PCA(n_components=2)
pca.fit(X_2d)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Original data with PCA directions
axes[0].scatter(X_2d[:, 0], X_2d[:, 1], alpha=0.5, s=20)
for length, vector, color, label in zip(
    pca.explained_variance_, pca.components_,
    ['red', 'blue'], ['PC1 (most variance)', 'PC2 (least variance)']
):
    v = vector * 3 * np.sqrt(length)
    axes[0].annotate('', arrowprops=dict(arrowstyle='->', color=color, lw=2.5),
                     xytext=(0, 0), xy=v)
axes[0].set_title('Original Data + PCA Directions')
axes[0].set_aspect('equal')

# Projected onto PC1 only
X_1d = pca.transform(X_2d)[:, 0]  # Keep only PC1
axes[1].scatter(X_1d, np.zeros_like(X_1d), alpha=0.5, s=20, color='green')
axes[1].set_title(f'Projected to 1D (PC1)\nCaptures {pca.explained_variance_ratio_[0]*100:.1f}% variance')
axes[1].set_xlabel('PC1')
axes[1].set_yticks([])

plt.tight_layout()
plt.show()

print(f'Variance explained by PC1: {pca.explained_variance_ratio_[0]*100:.1f}%')
print(f'Variance explained by PC2: {pca.explained_variance_ratio_[1]*100:.1f}%')

### Practical 2: Iris Dataset — 4D → 2D for Visualization

In [ ]:
iris = load_iris()
X = StandardScaler().fit_transform(iris.data)  # Always scale before PCA!

# Reduce to 2 components
pca2 = PCA(n_components=2)
X_pca = pca2.fit_transform(X)

plt.figure(figsize=(7, 5))
for cls, name, color in zip([0,1,2], iris.target_names, ['red','green','blue']):
    mask = iris.target == cls
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], label=name, color=color, s=40, alpha=0.8)

plt.title(f'Iris: 4 features → 2 PCA components\n'
          f'Variance explained: {sum(pca2.explained_variance_ratio_)*100:.1f}%')
plt.xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)')
plt.legend()
plt.tight_layout()
plt.show()

print('\n💡 With just 2 components we can see 3 species clearly separated!')

### Practical 3: Choosing Number of Components

In [ ]:
# How many components do we need to keep 95% of variance?
digits = load_digits()  # 64-feature dataset (8x8 pixel images)
X_digits = StandardScaler().fit_transform(digits.data)

pca_full = PCA().fit(X_digits)
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)
n_95 = np.argmax(cumulative_variance >= 0.95) + 1

plt.figure(figsize=(8, 4))
plt.plot(cumulative_variance, 'bo-', markersize=4)
plt.axhline(0.95, color='red', linestyle='--', label='95% threshold')
plt.axvline(n_95, color='green', linestyle='--', label=f'n={n_95} components needed')
plt.title('Cumulative Explained Variance (Digits Dataset: 64 features)')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Variance Explained')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Original features: 64')
print(f'Components needed for 95% variance: {n_95}')
print(f'Dimensionality reduction: {64 - n_95} fewer features ({(1 - n_95/64)*100:.0f}% reduction)')

### 🏋️ Try It Yourself!

In [ ]:
# ✏️ YOUR TURN: Change n_components and see reconstruction quality
n_components = 20  # Try: 5, 10, 20, 40, 64

pca_exp = PCA(n_components=n_components)
X_compressed = pca_exp.fit_transform(X_digits)
X_reconstructed = pca_exp.inverse_transform(X_compressed)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i in range(5):
    axes[0, i].imshow(digits.data[i].reshape(8,8), cmap='gray')
    axes[0, i].set_title('Original')
    axes[0, i].axis('off')
    
    axes[1, i].imshow(X_reconstructed[i].reshape(8,8), cmap='gray')
    axes[1, i].set_title(f'n={n_components}')
    axes[1, i].axis('off')

plt.suptitle(f'Original vs Reconstructed (n_components={n_components}, '
             f'variance={sum(pca_exp.explained_variance_ratio_)*100:.1f}%)')
plt.tight_layout()
plt.show()

## 🎓 Summary

| Concept | Takeaway |
|---------|----------|
| PCA | Finds directions of maximum variance |
| PC1 | Direction with MOST variance |
| Explained variance ratio | % of information each component carries |
| Rule of thumb | Keep components until you reach 95% cumulative variance |
| Always scale | PCA is sensitive to feature magnitude |

➡️ **Next: Model Evaluation & Cross-Validation**